# CLAUDE API MECHANICS HOMEWORK

<br>

**About:** Practice applying Messages API, streaming, batches, and cost tracking on different prompts and datasets from the lesson.

**Learning Goals:**
- Build a budget-capped multi-prompt pipeline using MessagesClient.
- Use streaming to count tokens delivered in real time.
- Reason about realtime vs batch API tradeoffs for concrete scenarios.
- Submit and track a Message Batches API job over a real dataset.
- Combine API mechanics with domain-appropriate reasoning at scale.

**Keywords:** MessagesClient, budget, streaming, batches, cost, custom_id

**Prerequisite Knowledge:** Lesson 01 complete

**Target User:** Developer who has completed 01_claude_api_mechanics.ipynb


**Estimated Cost to Run:** < $0.05 in default mode

**Estimated Completion:** 30-45 min

<hr style="border: 4px solid#003262;" />



#### CONTENTS

- [Q1: Budget-Capped Multi-Prompt Pipeline](#q1-budget-capped-multi-prompt-pipeline)
- [Q2: Streaming with Token Counting](#q2-streaming-with-token-counting)
- [Q3: Realtime vs Batch Decision Justification](#q3-realtime-vs-batch-decision-justification)
- [Q4: Batches API Round-Trip over Wikipedia Snippets](#q4-batches-api-round-trip-over-wikipedia-snippets)

<br>

In [1]:
# Paths in this repo are written relative to the repository root, but a
# Jupyter kernel starts in the notebook's own directory. Anchor to the root.
from claude_code.paths import use_repo_root
use_repo_root()

# Setup - imports and environment check
import os
import json
from pathlib import Path

import anthropic  # Verified against pypi.org/pypi/anthropic/json on 2026-09-02

from claude_code.api import MessagesClient, StreamingClient, BatchesClient, BatchRequest
from claude_code.api.errors import BudgetExceededError
from claude_code.config.model_registry import cost_for_call, get_model
from claude_code.security.secrets import load_api_key

DRY_RUN = os.getenv("CLAUDE_CODE_DRY_RUN", "0") == "1"
if DRY_RUN:
    print("DRY RUN: live API calls will be skipped")
else:
    _ = load_api_key("ANTHROPIC_API_KEY")

<hr style="border: 2px solid#003262;" />

## Q1: Budget-Capped Multi-Prompt Pipeline

Write a function `run_prompts(prompts: list[str], budget_usd: float) -> list[str]` that:
- Accepts a list of prompts and a shared cost cap in USD.

- Uses [`MessagesClient`](https://platform.claude.com/docs/en/build-with-claude/working-with-messages) with `haiku_current`.
  - `client.messages.create()` is a standard blocking request. It waits for the model to finish generating the entire response before returning a single, complete Message object.

- Calls each prompt in order.

- Stops (without raising) if the next call would exceed `budget_usd`.

- Returns a list of response text strings (one per successful call).

Store the results of running the function on the sample prompts below in `results`.

In [9]:
# Q1: Budget-capped multi-prompt pipeline
SAMPLE_PROMPTS = [
    "Name one animal that lives in the ocean. One word.",
    "Name one planet in our solar system. One word.",
    "Name one programming language. One word.",
    "Name one country in South America. One word.",
]

haiku_current = "claude-haiku-4-5-20251001"

In [ ]:
### YOUR CODE HERE ###
def run_prompts(prompts: list[str], budget_usd: float) -> list[str]:
    pass

results = []  # call run_prompts(SAMPLE_PROMPTS, 0.02) and store here

<details>
<summary>💡 Hints: Requirements-to-Code Translation </summary>

| Requirement | Code Translation / Component | Why it works |
| :--- | :--- | :--- |
| **Accepts a list of prompts and a shared cost cap in USD.** | `def run_prompts(prompts: list[str], budget_usd: float)` | Standard Python typing explicitly maps the collection of text blocks to a list and the financial constraint to a floating-point number. |
| **Uses `MessagesClient` with `haiku_current`.** | `client.messages.create(model="claude-3-5-haiku-20241022", ...)` | In the `anthropic` SDK, `client.messages` is an instance of the `MessagesClient` class. The string representation for the current `haiku` model is explicitly declared via the model parameter. |
| **Calls each prompt in order.** | `for prompt in prompts:` | A basic `for` loop guarantees sequential execution. It processes each string item in the list sequentially from index `0` to the end, ensuring synchronous execution. |
| **Stops (without raising) if the next call would exceed `budget_usd`.** | `if total_spent >= budget_usd: break` and `try... except: break` | Using a `break` statement exits the execution loop cleanly without raising a crashing exception. The logic computes token overhead against a tracking balance (`total_spent`) after each call. |
| **Returns a list of response text strings.** | `responses.append(response_text.strip())` followed by `return responses` | An internal collection list captures valid string outputs parsed from Anthropic's unstructured `message.content` array blocks and returns them seamlessly at completion. |

</details>

<hr style="border: 2px solid#003262;" />

## Q2: Streaming with Token Counting

Modify `run_prompts` from Q1 to use [`StreamingClient`](https://platform.claude.com/docs/en/build-with-claude/streaming) instead of [`MessagesClient`](https://platform.claude.com/docs/en/build-with-claude/working-with-messages). Collect each streamed chunk and count the total number of tokens streamed across all prompts.

* `client.messages.stream()` is a real-time streaming request. It immediately opens a server-sent events (SSE) connection and yields smaller chunks of text as they are being generated.

Store the total output token count in `total_tokens` (int).

Hint: Use `sc.last_call_tokens` -- that returns `(input_tokens, output_tokens)` after each stream ends.

In [ ]:
# Q2: Streaming with token counting
### YOUR CODE HERE ###
total_tokens = 0  # sum of output_tokens across all streaming calls

<hr style="border: 2px solid#003262;" />

## Q3: Realtime vs Batch Decision Justification

For each scenario below, decide whether to use the realtime Messages API or the Message Batches API. Justify each choice in a sentence.

| Scenario | Decision | Justification |
|---|---|---|
| A) A chatbot that answers user questions live in a web app | | |
| B) Nightly summarization of 10,000 log entries | | |
| C) A spell-checker that highlights errors as the user types | | |
| D) Generating weekly marketing copy for 500 product SKUs | | |


**YOUR ANSWERS HERE** - edit this cell to complete the table above.

| Scenario | Decision | Justification |
|---|---|---|
| A | | |
| B | | |
| C | | |
| D | | |

<hr style="border: 2px solid#003262;" />

## Q4: Batches API Round-Trip over Wikipedia Snippets

Using the fixture at `assets/homework_wikipedia_snippets.json` (10 article snippets), submit a Batches API job that asks Claude to summarize each snippet in one sentence.

1. Load the snippets from the JSON file.
2. Build one `BatchRequest` per snippet. Use the snippet's `id` as `custom_id`.
3. Submit the batch and store the batch handle in `batch_handle`.
4. (Optional - costs money) Poll to completion and store the results in `batch_results`.

In dry-run mode: print the request shapes and store the batch ID as `"dry-run-batch-id"` in `batch_id`.

* The purpose of a dry run is **to test a system, script, or process without committing resources, executing permanent changes, spending money, or risking system failure**.

  > ```python
  >if DRY_RUN:
  >    print("\n--- Dry Run Request Shapes ---")
  >    print(f"Total requests staged: {len(requests)}")
  >    print(f"Sample Custom ID: {requests[0]['custom_id']}")
  >    print(f"Sample Content Shape: {requests[0]['params']['messages'][0]['content'][:60]}...")
  >    
  >    batch_id = "dry-run-batch-id"
  >    batch_handle = None
  >else:
  >    # Submit the actual batch to the Anthropic API
  >    batch_handle = client.beta.messages.batches.create(requests=requests)
  >    batch_id = batch_handle.id
  >    print(f"Successfully submitted batch job with ID: {batch_id}")

In [ ]:
# Q4: Batches API round-trip over Wikipedia snippets
snippets_file = Path("assets/homework_wikipedia_snippets.json")
snippets = json.loads(snippets_file.read_text())
print(f"Loaded {len(snippets)} snippets")

Loaded 10 snippets


In [ ]:
# Enforces correct struture for every request appended to a bulk payloa
from anthropic.types.messages.batch_create_params import Request as BatchRequestParam

# Toggle this flag depending on whether you want a real run (False) or a dry-run (True) layout
DRY_RUN = True   


### YOUR CODE HERE ###


## Sources Consulted

- [Messages API Reference](https://platform.claude.com/docs/en/api/messages) - retrieved 2026-09-02; volatile
- [Message Batches API](https://platform.claude.com/docs/en/api/creating-message-batches) - retrieved 2026-09-02; volatile
- [Streaming Guide](https://platform.claude.com/docs/en/docs/build-with-claude/streaming) - retrieved 2026-09-02; volatile

<hr style="border: 6px solid#003262;" />